# Wikidata Experiments — R-GCN on tkgl-smallpedia

This notebook handles all Wikidata-specific logic:
- Dataset download and loading
- Model training (or checkpoint loading)
- Link prediction evaluation
- Interactive entity/relation prediction

The R-GCN model, training loop, and evaluation utilities live in `rgcn_model.py`.
Make sure it is in the same directory as this notebook.

---
## 0. Install & Imports

In [1]:
import sys
%pip install py-tgb torch-geometric
%pip install "numpy==1.26.4" --force-reinstall

  Obtaining dependency information for numpy<3.0.0,>=2.0.2 from https://files.pythonhosted.org/packages/de/2f/702a4594413c1a8632092beae8aba00f1d67947389369b3777aed783fdca/numpy-2.4.4-cp312-cp312-macosx_14_0_x86_64.whl.metadata
  Using cached numpy-2.4.4-cp312-cp312-macosx_14_0_x86_64.whl.metadata (6.6 kB)
Using cached numpy-2.4.4-cp312-cp312-macosx_14_0_x86_64.whl (6.6 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4

[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
  Obtaining dependency information for numpy==1.26.4 from https://files.pythonhosted.org/packages/95/12/8f2020a8e8b8383ac0177dc9570aad031a3beb12e38847f7129bacd96228/numpy-1.26.4-cp312-cp312-macosx_10_9_x86_64.whl.metadata
  Using cached numpy-1.26.4-cp312-cp312-macosx_10_9_x86_64.whl.metadata 

In [2]:
import os, json, pickle, time, zipfile
import requests
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

# ── Import everything from the model library ──────────────────────────────────
from rgcn_model import (
    DEVICE,
    RelationalGraph,
    RGCNLinkPredictor,
    RGCNLinkPredictorPyG,
    PYG_AVAILABLE,
    train,
    evaluate,
    plot_history,
)

print(f"PyTorch : {torch.__version__}")
print(f"Device  : {DEVICE}")
print(f"PyG     : {'available' if PYG_AVAILABLE else 'not installed'}")

/Users/Isaac/290-Final-Project/R-GCN/rgcn_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch : 2.2.2
Device  : cpu
PyG     : available


In [3]:
# ── Project folders ───────────────────────────────────────────────────────────
# Edit PROJECT_DIR if you want data/checkpoints stored somewhere else.
PROJECT_DIR    = os.getcwd()
DATA_DIR       = os.path.join(PROJECT_DIR, "datasets")
CHECKPOINT_DIR = os.path.join(PROJECT_DIR, "checkpoints")
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Data directory       : {DATA_DIR}")
print(f"Checkpoint directory : {CHECKPOINT_DIR}")

Data directory       : /Users/Isaac/290-Final-Project/R-GCN/datasets
Checkpoint directory : /Users/Isaac/290-Final-Project/R-GCN/checkpoints


---
## 1. Wikidata Dataset Loading

Downloads `tkgl-smallpedia` via `py-tgb` on first run and wraps it in a `RelationalGraph`.
English labels are fetched from the Wikidata API and cached progressively so long
runs can resume safely if interrupted.

In [4]:
# ── Constants ─────────────────────────────────────────────────────────────────
TGB_DATASET_NAME = "tkgl-smallpedia"
TGB_DATASET_DIR  = os.path.join(DATA_DIR, "tkgl_smallpedia")
LABEL_CACHE_PATH = os.path.join(DATA_DIR, "wikidata_labels_cache.json")
HEADERS          = {"User-Agent": "rgcn-project/1.0 (university research)"}


def save_label_cache(label_cache: dict) -> None:
    tmp_path = LABEL_CACHE_PATH + ".tmp"
    with open(tmp_path, "w") as f:
        json.dump(label_cache, f)
    os.replace(tmp_path, LABEL_CACHE_PATH)


def ensure_local_tgb_dataset(root: str = DATA_DIR) -> str:
    """Download tkgl-smallpedia into the local DATA_DIR if needed."""
    from tgb.utils.info import DATA_URL_DICT

    root = os.path.abspath(root)
    dataset_dir  = os.path.join(root, "tkgl_smallpedia")
    edgelist_path = os.path.join(dataset_dir, f"{TGB_DATASET_NAME}_edgelist.csv")

    if os.path.exists(edgelist_path):
        return dataset_dir

    os.makedirs(dataset_dir, exist_ok=True)
    zip_path = os.path.join(dataset_dir, f"{TGB_DATASET_NAME}.zip")
    url = DATA_URL_DICT[TGB_DATASET_NAME]

    print(f"Downloading {TGB_DATASET_NAME} into {dataset_dir}...")
    with requests.get(url, headers=HEADERS, stream=True, timeout=60) as resp:
        resp.raise_for_status()
        with open(zip_path, "wb") as f:
            for chunk in resp.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)

    with zipfile.ZipFile(zip_path, "r") as archive:
        archive.extractall(dataset_dir)
    os.remove(zip_path)

    if not os.path.exists(edgelist_path):
        raise FileNotFoundError(
            f"Download completed but {edgelist_path} was not found."
        )

    return dataset_dir


def fetch_wikidata_labels(
    ids: list,
    batch_size: int = 50,
    seed_cache: dict | None = None,
    save_every: int = 25,
    pause_seconds: float = 1.0,
    cooldown_every: int = 200,
    cooldown_seconds: float = 20.0,
) -> dict:
    label_cache = dict(seed_cache or {})
    if not ids:
        return label_cache

    total_batches = (len(ids) + batch_size - 1) // batch_size
    session = requests.Session()
    session.headers.update(HEADERS)

    for batch_num, start in enumerate(range(0, len(ids), batch_size), start=1):
        batch = ids[start : start + batch_size]

        if batch_num > 1 and (batch_num - 1) % cooldown_every == 0:
            print(
                f"Cooling down for {cooldown_seconds:.0f}s before batch "
                f"{batch_num}/{total_batches}..."
            )
            time.sleep(cooldown_seconds)

        for attempt in range(5):
            try:
                resp = session.get(
                    "https://www.wikidata.org/w/api.php",
                    params={
                        "action": "wbgetentities",
                        "ids": "|".join(batch),
                        "props": "labels",
                        "languages": "en",
                        "format": "json",
                    },
                    timeout=30,
                )

                if resp.status_code == 429:
                    retry_after = resp.headers.get("Retry-After")
                    wait = max(float(retry_after) if retry_after else 0.0,
                               min(120.0, 5.0 * (2 ** attempt)))
                    print(
                        f"  Batch {batch_num}/{total_batches} was rate-limited; "
                        f"waiting {wait:.0f}s before retrying..."
                    )
                    time.sleep(wait)
                    continue

                resp.raise_for_status()
                entities = resp.json().get("entities", {})
                for qid in batch:
                    info = entities.get(qid, {})
                    label_cache[qid] = info.get("labels", {}).get("en", {}).get("value", qid)
                break
            except (requests.RequestException, ValueError) as e:
                wait = min(120.0, 5.0 * (2 ** attempt))
                print(
                    f"  Batch {batch_num}/{total_batches} attempt {attempt+1} failed ({e}); "
                    f"retrying in {wait:.0f}s..."
                )
                time.sleep(wait)
        else:
            print(f"  Batch {batch_num}/{total_batches} failed repeatedly; keeping raw IDs.")
            for qid in batch:
                label_cache.setdefault(qid, qid)

        if batch_num % save_every == 0 or batch_num == total_batches:
            save_label_cache(label_cache)
            print(
                f"Saved {len(label_cache):,} labels after batch "
                f"{batch_num}/{total_batches}."
            )

        time.sleep(pause_seconds)

    return label_cache


def load_wikidata(root: str = DATA_DIR) -> RelationalGraph:
    """
    Load tkgl-smallpedia via TGB and wrap it in a RelationalGraph.

    - Downloads raw TGB files into DATA_DIR if needed.
    - English labels are fetched from the Wikidata API on first run and
      cached progressively so long runs can resume safely.
    """
    import tgb.linkproppred.dataset as tgb_dataset_module
    from tgb.linkproppred.dataset import LinkPropPredDataset

    root = os.path.abspath(root)
    ensure_local_tgb_dataset(root)

    original_proj_dir = tgb_dataset_module.PROJ_DIR
    tgb_dataset_module.PROJ_DIR = ""
    try:
        dataset = LinkPropPredDataset(
            name=TGB_DATASET_NAME,
            root=root,
            preprocess=True,
            download=False,
        )
    finally:
        tgb_dataset_module.PROJ_DIR = original_proj_dir

    data = dataset.full_data

    src   = torch.from_numpy(data["sources"].astype(np.int64))
    dst   = torch.from_numpy(data["destinations"].astype(np.int64))
    rel   = torch.from_numpy(data["edge_type"].astype(np.int64))
    edges = torch.stack([src, dst, rel], dim=1)

    train_mask = torch.from_numpy(dataset.train_mask)
    val_mask   = torch.from_numpy(dataset.val_mask)
    test_mask  = torch.from_numpy(dataset.test_mask)

    tgb_dir = dataset.root
    print(f"Using TGB files from: {tgb_dir}")

    with open(os.path.join(tgb_dir, "ml_tkgl-smallpedia_nodeid.pkl"), "rb") as f:
        node2int = pickle.load(f)
    int2node = {v: k for k, v in node2int.items()}

    raw         = pd.read_csv(dataset.meta_dict["fname"])
    unique_rels = list(dict.fromkeys(raw["relation_type"]))
    int2rel     = {i: p for i, p in enumerate(unique_rels)}

    # Load or fetch English labels
    if os.path.exists(LABEL_CACHE_PATH):
        print(f"Loading label cache from {LABEL_CACHE_PATH}")
        with open(LABEL_CACHE_PATH) as f:
            label_cache = json.load(f)
    else:
        label_cache = {}

    all_ids = list(set(int2node.values()) | set(int2rel.values()))
    missing = [wid for wid in all_ids if wid not in label_cache]
    if missing:
        print(f"Fetching {len(missing):,} Wikidata labels (first run only — "
              f"results cached to {LABEL_CACHE_PATH})...")
        label_cache = fetch_wikidata_labels(missing, seed_cache=label_cache)
        save_label_cache(label_cache)
        print("Labels cached.")

    node_labels = {i: label_cache.get(q, q) for i, q in int2node.items()}
    rel_labels  = {i: label_cache.get(p, p) for i, p in int2rel.items()}

    return RelationalGraph(
        name          = "tkgl-smallpedia",
        num_nodes     = dataset.num_nodes,
        num_relations = dataset.num_rels,
        train_edges   = edges[train_mask],
        val_edges     = edges[val_mask],
        test_edges    = edges[test_mask],
        node_labels   = node_labels,
        rel_labels    = rel_labels,
    )

In [5]:
# ── Checkpoint paths ──────────────────────────────────────────────────────────
SCRATCH_CKPT = os.path.join(CHECKPOINT_DIR, "rgcn_scratch_wikidata.pt")
PYG_CKPT     = os.path.join(CHECKPOINT_DIR, "rgcn_pyg_wikidata.pt")
GRAPH_CKPT   = os.path.join(CHECKPOINT_DIR, "wiki_graph.pkl")

# Load graph from checkpoint if available, otherwise load from TGB
if os.path.exists(GRAPH_CKPT):
    print(f"Loading graph from checkpoint: {GRAPH_CKPT}")
    with open(GRAPH_CKPT, "rb") as f:
        wiki_graph = pickle.load(f)
    print("Graph loaded from checkpoint — skipping TGB processing.")
else:
    print("No graph checkpoint found — loading from TGB...")
    wiki_graph = load_wikidata()
    with open(GRAPH_CKPT, "wb") as f:
        pickle.dump(wiki_graph, f)
    print(f"Graph saved to {GRAPH_CKPT}")

wiki_graph.summary()

Loading graph from checkpoint: /Users/Isaac/290-Final-Project/R-GCN/checkpoints/wiki_graph.pkl
Graph loaded from checkpoint — skipping TGB processing.
RelationalGraph: tkgl-smallpedia
  Nodes         : 47,433
  Relation types: 566
  Train edges   : 775,514
  Val edges     : 162,066
  Test edges    : 163,172
  Node features : no (will use learned embeddings)


In [6]:
# Verify label samples
print("Sample node labels:")
print(list(wiki_graph.node_labels.items())[:5])
print("\nSample relation labels:")
print(list(wiki_graph.rel_labels.items())[:5])

Sample node labels:
[(0, 'Lille'), (1, 'Legion of Honour'), (2, 'Copley Medal'), (3, 'Marcellin Berthelot'), (4, 'Bukharan Jews')]

Sample relation labels:
[(0, 'award received'), (1, 'winner'), (2, 'country'), (3, 'doctoral student'), (4, 'significant event')]


---
## 2. From-Scratch Model: Train or Load

In [7]:
if torch.cuda.is_available():
    torch.cuda.empty_cache()

wiki_model_scratch = RGCNLinkPredictor(
    num_nodes     = wiki_graph.num_nodes,
    num_relations = wiki_graph.num_relations,
    hidden_dim    = 64,
    num_layers    = 2,
    num_bases     = 30,
    dropout       = 0.1,
)
print(f"Scratch model parameters: {sum(p.numel() for p in wiki_model_scratch.parameters()):,}")

if os.path.exists(SCRATCH_CKPT):
    print(f"Checkpoint found — loading scratch model from {SCRATCH_CKPT}")
    wiki_model_scratch.load_state_dict(torch.load(SCRATCH_CKPT, map_location=DEVICE))
    wiki_model_scratch.to(DEVICE).eval()
    wiki_history_scratch = None
    print("Scratch model ready (training skipped).")
else:
    print("No checkpoint found — training scratch model...")
    wiki_history_scratch = train(
        wiki_model_scratch, wiki_graph,
        num_epochs=50, lr=1e-3, batch_size=4096, eval_every=5,
    )
    torch.save(wiki_model_scratch.state_dict(), SCRATCH_CKPT)
    print(f"Scratch model saved to {SCRATCH_CKPT}")

Scratch model parameters: 3,359,848
Checkpoint found — loading scratch model from /Users/Isaac/290-Final-Project/R-GCN/checkpoints/rgcn_scratch_wikidata.pt
Scratch model ready (training skipped).


In [8]:
if wiki_history_scratch is not None:
    plot_history(wiki_history_scratch, "Wikidata — R-GCN from scratch")
else:
    print("Model was loaded from checkpoint — no training history to plot.")

Model was loaded from checkpoint — no training history to plot.


---
## 3. From-Scratch Model: Test Evaluation

In [9]:
train_e    = wiki_graph.train_edges.to(DEVICE)
edge_index = train_e[:, :2].t().contiguous()
edge_type  = train_e[:, 2]

test_metrics = evaluate(
    wiki_model_scratch, wiki_graph,
    wiki_graph.test_edges, edge_index, edge_type
)
print("Test results (from-scratch model):")
for k, v in test_metrics.items():
    print(f"  {k:10s}: {v:.4f}")

Test results (from-scratch model):
  mrr       : 0.2897
  hits@1    : 0.2026
  hits@3    : 0.2684
  hits@10   : 0.4694


---
## 4. PyG Model: Train or Load

In [19]:
TRAIN_PYG = False 

if PYG_AVAILABLE and TRAIN_PYG:
    wiki_model_pyg = RGCNLinkPredictorPyG(
        num_nodes     = wiki_graph.num_nodes,
        num_relations = wiki_graph.num_relations,
        hidden_dim    = 64,
        num_layers    = 2,
        num_bases     = 30,
        dropout       = 0.1,
    )
    print(f"PyG model parameters: {sum(p.numel() for p in wiki_model_pyg.parameters()):,}")

    if os.path.exists(PYG_CKPT):
        print(f"Checkpoint found — loading PyG model from {PYG_CKPT}")
        wiki_model_pyg.load_state_dict(torch.load(PYG_CKPT, map_location=DEVICE))
        wiki_model_pyg.to(DEVICE).eval()
        wiki_history_pyg = None
        print("PyG model ready (training skipped).")
    else:
        print("No PyG checkpoint found — training...")
        wiki_history_pyg = train(
            wiki_model_pyg, wiki_graph,
            num_epochs=50, lr=1e-3, batch_size=4096, eval_every=5,
        )
        torch.save(wiki_model_pyg.state_dict(), PYG_CKPT)
        print(f"PyG model saved to {PYG_CKPT}")
else:
    print("PyG training skipped (TRAIN_PYG=False). Set TRAIN_PYG=True to enable.")
    wiki_model_pyg = None
    wiki_history_pyg = None

PyG training skipped (TRAIN_PYG=False). Set TRAIN_PYG=True to enable.


In [20]:
if PYG_AVAILABLE and TRAIN_PYG:
    if wiki_history_pyg is not None:
        plot_history(wiki_history_pyg, "Wikidata — R-GCN (PyG)")
    else:
        print("PyG model was loaded from checkpoint — no training history to plot.")

---
## 5. Scratch vs PyG Comparison

In [21]:
if PYG_AVAILABLE and wiki_history_scratch is not None and wiki_history_pyg is not None:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle("Wikidata: Scratch vs PyG", fontsize=13)
    for hist, label, ls in [
        (wiki_history_scratch, 'Scratch', '-'),
        (wiki_history_pyg,     'PyG',     '--')
    ]:
        axes[0].plot(hist['train_loss'], ls=ls, label=label)
        axes[1].plot(hist['val_mrr'],    ls=ls, label=f"{label} MRR")
        axes[1].plot(hist['val_hits10'], ls=ls, alpha=0.6, label=f"{label} Hits@10")
    axes[0].set_title('Loss'); axes[0].legend()
    axes[1].set_title('Val Metrics'); axes[1].legend()
    plt.tight_layout(); plt.show()
else:
     print("Comparison plot skipped — need both models trained in the same session.")

Comparison plot skipped — need both models trained in the same session.


---
## 6. Link Prediction Inference

Pre-compute node embeddings once, then use them for `predict_object` and `predict_relation`.

In [11]:
# ── Pre-compute node embeddings ───────────────────────────────────────────────
# Swap wiki_model_pyg for wiki_model_scratch here if you prefer.
train_e    = wiki_graph.train_edges.to(DEVICE)
edge_index = train_e[:, :2].t().contiguous()
edge_type  = train_e[:, 2]

active_model = wiki_model_scratch

with torch.no_grad():
    node_emb = active_model.encoder(
        edge_index, edge_type,
        num_nodes=wiki_graph.num_nodes
    )

# Alias used by predict_object / predict_relation
model = active_model
print(f"Node embeddings shape: {node_emb.shape}")

Node embeddings shape: torch.Size([47433, 64])


In [12]:
def predict_object(subject: str, relation: str, top_k: int = 10):
    """
    Given a subject entity and relation (as English strings),
    return the top_k most likely object entities.
    """
    node2int = {v: k for k, v in wiki_graph.node_labels.items()}
    rel2int  = {v: k for k, v in wiki_graph.rel_labels.items()}

    s_idx, r_idx = None, None

    if subject not in node2int:
        print(f"Subject '{subject}' not found in node labels.")
        matches = [n for n in node2int if subject.lower() in n.lower()]
        if matches:
            print(f"  Partial matches ({len(matches)} found, showing first 5):",
                  ", ".join(matches[:5]))
        else:
            print("  No partial matches found.")
        return
    else:
        s_idx = node2int[subject]

    if relation not in rel2int:
        print(f"Relation '{relation}' not found in relation labels.")
        matches = [r for r in rel2int if relation.lower() in r.lower()]
        if matches:
            print(f"  Partial matches ({len(matches)} found, showing first 5):",
                  ", ".join(matches[:5]))
        else:
            print("  No partial matches found.")
        return
    else:
        r_idx = rel2int[relation]

    with torch.no_grad():
        h_s = node_emb[s_idx].unsqueeze(0)
        r   = model.decoder.relation_emb(
                  torch.tensor([r_idx], device=DEVICE))
        scores = (h_s * r * node_emb).sum(dim=-1)

    top_scores, top_indices = scores.topk(top_k)

    print(f"\n  '{subject}'  --[{relation}]-->  ???\n")
    print(f"  {'Rank':<6} {'Entity':<35} {'Score':>8}")
    print("  " + "-" * 52)
    for rank, (idx, score) in enumerate(zip(top_indices.tolist(),
                                             top_scores.tolist()), 1):
        label = wiki_graph.node_labels.get(idx, str(idx))
        print(f"  {rank:<6} {label:<35} {score:>8.4f}")


def predict_relation(subject: str, obj: str, top_k: int = 5):
    """
    Given subject and object entities, return the most likely relations.
    """
    node2int = {v: k for k, v in wiki_graph.node_labels.items()}

    s_idx, o_idx = None, None

    if subject not in node2int:
        print(f"Subject '{subject}' not found in node labels.")
        matches = [n for n in node2int if subject.lower() in n.lower()]
        if matches:
            print(f"  Partial matches ({len(matches)} found, showing first 5):",
                  ", ".join(matches[:5]))
        return
    else:
        s_idx = node2int[subject]

    if obj not in node2int:
        print(f"Object '{obj}' not found in node labels.")
        matches = [n for n in node2int if obj.lower() in n.lower()]
        if matches:
            print(f"  Partial matches ({len(matches)} found, showing first 5):",
                  ", ".join(matches[:5]))
        return
    else:
        o_idx = node2int[obj]

    with torch.no_grad():
        h_s   = node_emb[s_idx]
        h_o   = node_emb[o_idx]
        all_r = model.decoder.relation_emb.weight
        scores = (h_s * all_r * h_o).sum(dim=-1)

    top_scores, top_indices = scores.topk(top_k)

    print(f"\n  '{subject}'  --[???]-->  '{obj}'\n")
    print(f"  {'Rank':<6} {'Relation':<35} {'Score':>8}")
    print("  " + "-" * 52)
    for rank, (idx, score) in enumerate(zip(top_indices.tolist(),
                                             top_scores.tolist()), 1):
        label = wiki_graph.rel_labels.get(idx, str(idx))
        print(f"  {rank:<6} {label:<35} {score:>8.4f}")

---
## 7. Example Predictions

In [13]:
# Predict what awards Lille has received
predict_object("Lille", "award received")

# Predict who won the Copley Medal
predict_object("Copley Medal", "winner")

# Predict the relation between two known entities
predict_relation("Ferdinand von Lindemann", "Martin Wilhelm Kutta")

# Predict doctoral students of a mathematician
predict_object("Felix Klein", "doctoral student")


  'Lille'  --[award received]-->  ???

  Rank   Entity                                 Score
  ----------------------------------------------------
  1      Legion of Honour                     20.3098
  2      Scharnhorst Order                     8.6897
  3      Peace Prize of the German Publishers' and Booksellers' Association   7.7333
  4      Lasker-Bloomberg Public Service Award   5.6158
  5      Reinhold Schneider Prize              5.1485
  6      Newbery Medal                         3.8426
  7      Royal Gold Medal                      3.7913
  8      Henry Draper Medal                    3.4661
  9      People's Hero of Yugoslavia           3.3694
  10     Order of St. Olav                     3.3181

  'Copley Medal'  --[winner]-->  ???

  Rank   Entity                                 Score
  ----------------------------------------------------
  1      Michael Atiyah                       17.0621
  2      Ray Lankester                        15.7003
  3      Frederick Hop

---
## 8. Graph Exploration Utilities

In [14]:
def entity_stats(name: str):
    """Show how many training edges an entity appears in as subject/object."""
    node2int = {v: k for k, v in wiki_graph.node_labels.items()}
    if name not in node2int:
        print(f"Entity '{name}' not found in node labels.")
        matches = [n for n in node2int if name.lower() in n.lower()]
        if matches:
            print(f"  Partial matches ({len(matches)} found, showing first 5):",
                  ", ".join(matches[:5]))
        return

    idx = node2int[name]
    as_subject = wiki_graph.train_edges[wiki_graph.train_edges[:, 0] == idx]
    as_object  = wiki_graph.train_edges[wiki_graph.train_edges[:, 1] == idx]
    print(f"'{name}' — {len(as_subject)} outgoing, {len(as_object)} incoming training edges")


# Example
entity_stats("Felix Klein")

'Felix Klein' — 1 outgoing, 1 incoming training edges


In [15]:
# Relations with enough data for reliable prediction
print("Relations with >1000 training edges:")
for r_idx in range(wiki_graph.num_relations):
    count = (wiki_graph.train_edges[:, 2] == r_idx).sum().item()
    if count > 1000:
        label = wiki_graph.rel_labels.get(r_idx, str(r_idx))
        print(f"  {label:<35} {count:>6,}")

Relations with >1000 training edges:
  award received                       8,585
  winner                               1,442
  country                             34,892
  significant event                    1,822
  work location                        6,724
  occupation                           3,230
  country of citizenship              85,508
  located in the administrative territorial entity 52,248
  capital of                           2,629
  named after                          2,159
  contains the administrative territorial entity  2,499
  shares border with                   1,053
  spouse                              21,621
  operator                             4,127
  headquarters location                1,310
  noble title                          1,188
  member of                            5,914
  employer                            21,823
  owned by                             1,304
  chairperson                          2,801
  head of state                        

In [16]:
# See all valid relation types
for idx, label in sorted(wiki_graph.rel_labels.items(), key=lambda x: x[1]):
    print(label)

IUCN conservation status
academic appointment
academic degree
academic thesis
adjacent station
affiliation
allegiance
anthem
applies to jurisdiction
architect
archives at
assessment
astronaut mission
audio system
award received
based on
basic form of government
board member
brand
broadcast by
business model
candidate
canonization status
capital
capital of
carries thoroughfare
cast member
cause of death
cause of destruction
chairperson
charge
charted in
chief executive officer
child organization or unit
clerked for
co-driver
coach of sports team
collection
colonel-in-chief
color
commanded by
commander of (DEPRECATED)
competition won
composer
connecting line
connecting service
consecrator
contains
contains settlement
contains the administrative territorial entity
contributed to creative work
convicted of
copyright holder
corporate officer
costume designer
country
country for sport
country of citizenship
country of origin
country of registry
cover art by
creator
crew member
currency
day i

---
## 9. Manual Checkpoint Management

Cells below are commented out — uncomment only if you need to force-save or force-reload.

In [ ]:
# Force save
# torch.save(wiki_model_scratch.state_dict(), SCRATCH_CKPT)
# print(f"Scratch model saved to {SCRATCH_CKPT}")
#
# if PYG_AVAILABLE:
#     torch.save(wiki_model_pyg.state_dict(), PYG_CKPT)
#     print(f"PyG model saved to {PYG_CKPT}")

In [ ]:
# Force reload
# wiki_model_scratch.load_state_dict(torch.load(SCRATCH_CKPT, map_location=DEVICE))
# wiki_model_scratch.to(DEVICE).eval()
# print("Scratch model reloaded.")

In [ ]:
# Copy an existing wikidata_labels_cache.json from a teammate into DATA_DIR:
# import shutil
# shutil.copy("wikidata_labels_cache.json", LABEL_CACHE_PATH)